# NSTT — SLR54 Data Preparation (T-002)

Downloads [OpenSLR SLR54](https://www.openslr.org/54/), preprocesses audio (16 kHz mono, duration filter), NFC-normalizes transcripts, and produces speaker-disjoint 80/10/10 manifests.

**Prerequisites:** Run `00_setup.ipynb` first. Store the project on Google Drive so raw/processed audio persist across sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

# Adjust if your Drive path differs
PROJECT_ROOT = Path("/content/drive/MyDrive/NSTT")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/NSTT")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# Optional: limit to one zip for a quick smoke test; set to None for full corpus (~8 GB)
ZIP_SUBSET = None  # e.g. ["asr_nepali_0.zip"] for smoke test
SEED = 42

In [ ]:
from src.pipeline import run_slr54_pipeline

result = run_slr54_pipeline(
    PROJECT_ROOT,
    zip_names=ZIP_SUBSET,
    seed=SEED,
)

stats = result["stats"]
print("Pipeline stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print("Manifests written to:", result["manifest_paths"])

In [ ]:
# Spot-check a few manifest rows and processed audio files
import soundfile as sf
from src.manifests import read_jsonl_manifest

manifest_dir = PROJECT_ROOT / "data" / "manifests"
for split in ["train", "val", "test"]:
    rows = read_jsonl_manifest(manifest_dir / f"{split}.jsonl")
    print(f"{split}: {len(rows)} utterances")
    for row in rows[:2]:
        audio = PROJECT_ROOT / row["audio_path"]
        data, sr = sf.read(audio)
        ch = 1 if data.ndim == 1 else data.shape[1]
        print(f"  {row['utterance_id']}: sr={sr}, channels={ch}, dur={row['duration_s']}s")